# Model C — TTA embeddings + CE

Overlapping-window embeddings at inference time. Kaggle public LB: 0.798.


## Colab setup

1. Place `repro/` inside your Drive project: `BirdCLEF_Project/repro/`.
2. Open any notebook from that Drive folder in Colab (right-click → Open with → Colab).
3. Runtime: **GPU** (T4 or better).
4. Colab secrets: add `KAGGLE_API_TOKEN` (key icon in the left sidebar).
5. `BirdCLEF_Project/` on Drive must also contain:
   - `perch_v2_no_dft.onnx`
   - `embeddings_v2_archive.zip` (training notebooks)
   - `embeddings_v2_TTA_archive.zip` (TTA / final model notebooks)
   - `train.csv` and `sample_submission.csv` (or download in notebook 02)

The first code cell mounts Drive and loads `birdclef` from `BirdCLEF_Project/repro/`.
Embeddings are unzipped to `/content/` for fast GPU training; checkpoints save to `repro/outputs/`.

After execution, download the notebook with outputs and re-upload for the GitHub delivery pass.


In [1]:
from google.colab import drive

import os
import sys
from pathlib import Path

drive.mount("/content/drive")

DRIVE_PROJECT = Path("/content/drive/MyDrive/BirdCLEF_Project")
REPRO_ROOT = DRIVE_PROJECT / "repro"

if not (REPRO_ROOT / "birdclef").exists():
    raise FileNotFoundError(
        f"Expected repro at {REPRO_ROOT}. Place repro inside BirdCLEF_Project on Drive."
    )

sys.path.insert(0, str(REPRO_ROOT))
os.chdir(REPRO_ROOT)
print(f"Working directory: {REPRO_ROOT}")

!pip install -q onnxscript onnxruntime-gpu soundfile tqdm scikit-learn

from birdclef.colab import colab_prepare_training

repro_root = colab_prepare_training(stage_tta=True)
print(f"Project root: {repro_root}")


Mounted at /content/drive
Working directory: /content/drive/MyDrive/BirdCLEF_Project/repro
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.0/277.0 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 96.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 15.6 MB/s eta 0:00:00
Copied train.csv to /content/train.csv
Copied sample_submission.csv to /content/sample_submission.csv
Unzipped baseline embeddings to /content/embeddings_v2
Unzipped TTA embeddings to /content/embeddings_v2_TTA
Project root: /content/drive/MyDrive/BirdCLEF_Project/repro


In [2]:
import os
import json
import warnings
import logging
from contextlib import redirect_stdout, redirect_stderr

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

from birdclef import (
    BirdDataset,
    BirdMoE,
    FocalLoss,
    competition_macro_roc_auc,
    plot_and_save_learning_curves,
    prepare_baseline_data,
    prepare_tta_data,
    seed_everything,
)
from birdclef.paths import MODELS_DIR

seed_everything(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on device: {device}")


Training on device: cuda


In [3]:
df_TTA, NUM_CLASSES, _ = prepare_tta_data()

In [4]:
SAVE_DIR = MODELS_DIR / "tta_ce_10ep"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

EPOCHS = 10
tta_fold_scores = []

for TRAIN_FOLD in range(5):
    best_auc = 0.0
    train_df = df_TTA[df_TTA["fold"] != TRAIN_FOLD].reset_index(drop=True)
    valid_df = df_TTA[df_TTA["fold"] == TRAIN_FOLD].reset_index(drop=True)
    train_ds = BirdDataset(train_df, is_tta=True)
    valid_ds = BirdDataset(valid_df, is_tta=True)
    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=0)
    valid_loader = DataLoader(valid_ds, batch_size=64, shuffle=False, num_workers=0)
    fold_train_losses = []
    fold_val_losses = []
    fold_val_aucs = []

    print(f"TRAINING FOLD {TRAIN_FOLD}")
    print(f"Training on {len(train_df)} samples, validating on {len(valid_df)} samples")

    # Fresh model/optimizer per fold so each fold is an independent CV estimate.
    seed_everything(42)
    model = BirdMoE(input_dim=1536, num_classes=NUM_CLASSES).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    for epoch in range(EPOCHS):
        model.train()
        train_loss = 0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()
            train_loss += loss.item()
        avg_train_loss = train_loss / len(train_loader)

        model.eval()
        val_loss = 0
        all_preds, all_labels = [], []
        with torch.no_grad():
            for inputs, labels in valid_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                all_preds.append(F.softmax(outputs, dim=1).cpu().numpy())
                all_labels.append(labels.cpu().numpy())
        avg_val_loss = val_loss / len(valid_loader)
        val_preds = np.concatenate(all_preds)
        val_labels = np.concatenate(all_labels)
        val_labels_onehot = np.eye(NUM_CLASSES)[val_labels]
        current_auc = competition_macro_roc_auc(val_labels_onehot, val_preds)
        print(
            f"Epoch {epoch + 1:02d}/{EPOCHS} | "
            f"Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val AUC: {current_auc:.4f}"
        )
        fold_train_losses.append(avg_train_loss)
        fold_val_losses.append(avg_val_loss)
        fold_val_aucs.append(current_auc)
        if current_auc > best_auc:
            best_auc = current_auc
            save_path = SAVE_DIR / f"best_moe_fold{TRAIN_FOLD}.pth"
            torch.save(model.state_dict(), save_path)
            print(f"Model saved to {save_path}")

    onnx_save_path = os.path.join(str(SAVE_DIR), f"bird_moe_fold{TRAIN_FOLD}.onnx")
    plot_and_save_learning_curves(fold_train_losses, fold_val_losses, fold_val_aucs, TRAIN_FOLD, str(SAVE_DIR))
    model.load_state_dict(torch.load(save_path, map_location=device))
    model.eval()
    dummy_input = torch.randn(1, 1536).to(device)
    with open(os.devnull, "w") as f, redirect_stdout(f), redirect_stderr(f), warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for name in ("onnx", "onnxruntime", "torch.onnx", "onnxscript"):
            logging.getLogger(name).setLevel(logging.CRITICAL)
        torch.onnx.export(
            model,
            dummy_input,
            onnx_save_path,
            export_params=True,
            opset_version=15,
            do_constant_folding=True,
            input_names=["embedding"],
            output_names=["logits"],
            dynamic_axes={"embedding": {0: "batch_size"}, "logits": {0: "batch_size"}},
        )
    print(f"Exported ONNX model to {onnx_save_path}")
    tta_fold_scores.append(best_auc)

print(f"CV score: {np.mean(tta_fold_scores):.4f} (+/- {np.std(tta_fold_scores):.4f})")

100%|██████████| 7110/7110 [00:09<00:00, 762.82it/s] 


TRAINING FOLD 0
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.9640 | Val Loss: 1.4545 | Val AUC: 0.9765
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold0.pth
Epoch 02/10 | Train Loss: 0.7349 | Val Loss: 1.4850 | Val AUC: 0.9747
Epoch 03/10 | Train Loss: 0.6823 | Val Loss: 1.5078 | Val AUC: 0.9757
Epoch 04/10 | Train Loss: 0.6468 | Val Loss: 1.5259 | Val AUC: 0.9762
Epoch 05/10 | Train Loss: 0.6242 | Val Loss: 1.4899 | Val AUC: 0.9769
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold0.pth
Epoch 06/10 | Train Loss: 0.6089 | Val Loss: 1.5384 | Val AUC: 0.9750
Epoch 07/10 | Train Loss: 0.5956 | Val Loss: 1.5459 | Val AUC: 0.9760
Epoch 08/10 | Train Loss: 0.5881 | Val Loss: 1.5532 | Val AUC: 0.9758
Epoch 09/10 | Train Loss: 0.5801 | Val Loss: 1.5381 | Val AUC: 0.9757
Epoch 10/10 | Train Loss: 0.5736 | Val Loss: 1.5167 | Val AUC: 0.9764
Learning curves 

100%|██████████| 7110/7110 [00:09<00:00, 715.15it/s]


TRAINING FOLD 1
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.9599 | Val Loss: 1.4616 | Val AUC: 0.9756
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold1.pth
Epoch 02/10 | Train Loss: 0.7364 | Val Loss: 1.4595 | Val AUC: 0.9759
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold1.pth
Epoch 03/10 | Train Loss: 0.6822 | Val Loss: 1.4671 | Val AUC: 0.9767
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold1.pth
Epoch 04/10 | Train Loss: 0.6459 | Val Loss: 1.4721 | Val AUC: 0.9775
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold1.pth
Epoch 05/10 | Train Loss: 0.6176 | Val Loss: 1.4700 | Val AUC: 0.9770
Epoch 06/10 | Train Loss: 0.5992 | Val Loss: 1.4908 | Val AUC: 0.9771
Epoch 07/10 | Train Loss: 0.5868 | Val Loss: 1.4749 | Val AUC: 0.9770
Epoch 08/10 

100%|██████████| 7110/7110 [00:11<00:00, 644.54it/s]


TRAINING FOLD 2
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.9631 | Val Loss: 1.4407 | Val AUC: 0.9758
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold2.pth
Epoch 02/10 | Train Loss: 0.7340 | Val Loss: 1.4361 | Val AUC: 0.9760
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold2.pth
Epoch 03/10 | Train Loss: 0.6849 | Val Loss: 1.4319 | Val AUC: 0.9754
Epoch 04/10 | Train Loss: 0.6559 | Val Loss: 1.4658 | Val AUC: 0.9743
Epoch 05/10 | Train Loss: 0.6329 | Val Loss: 1.4848 | Val AUC: 0.9744
Epoch 06/10 | Train Loss: 0.6157 | Val Loss: 1.4734 | Val AUC: 0.9754
Epoch 07/10 | Train Loss: 0.6027 | Val Loss: 1.4638 | Val AUC: 0.9757
Epoch 08/10 | Train Loss: 0.5911 | Val Loss: 1.4889 | Val AUC: 0.9738
Epoch 09/10 | Train Loss: 0.5809 | Val Loss: 1.4779 | Val AUC: 0.9759
Epoch 10/10 | Train Loss: 0.5762 | Val Loss: 1.4741 | Val AUC: 0.9759
Learning curves 

100%|██████████| 7110/7110 [00:11<00:00, 634.92it/s]


TRAINING FOLD 3
Training on 28439 samples, validating on 7110 samples
Epoch 01/10 | Train Loss: 0.9584 | Val Loss: 1.4656 | Val AUC: 0.9760
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold3.pth
Epoch 02/10 | Train Loss: 0.7313 | Val Loss: 1.4810 | Val AUC: 0.9758
Epoch 03/10 | Train Loss: 0.6788 | Val Loss: 1.4892 | Val AUC: 0.9770
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold3.pth
Epoch 04/10 | Train Loss: 0.6511 | Val Loss: 1.5120 | Val AUC: 0.9753
Epoch 05/10 | Train Loss: 0.6312 | Val Loss: 1.4908 | Val AUC: 0.9741
Epoch 06/10 | Train Loss: 0.6137 | Val Loss: 1.5394 | Val AUC: 0.9731
Epoch 07/10 | Train Loss: 0.6025 | Val Loss: 1.5344 | Val AUC: 0.9732
Epoch 08/10 | Train Loss: 0.5916 | Val Loss: 1.4923 | Val AUC: 0.9761
Epoch 09/10 | Train Loss: 0.5824 | Val Loss: 1.5137 | Val AUC: 0.9745
Epoch 10/10 | Train Loss: 0.5732 | Val Loss: 1.5011 | Val AUC: 0.9770
Learning curves 

100%|██████████| 7109/7109 [00:10<00:00, 686.27it/s]


TRAINING FOLD 4
Training on 28440 samples, validating on 7109 samples
Epoch 01/10 | Train Loss: 0.9647 | Val Loss: 1.4183 | Val AUC: 0.9768
Model saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/best_moe_fold4.pth
Epoch 02/10 | Train Loss: 0.7343 | Val Loss: 1.4472 | Val AUC: 0.9760
Epoch 03/10 | Train Loss: 0.6802 | Val Loss: 1.4493 | Val AUC: 0.9763
Epoch 04/10 | Train Loss: 0.6498 | Val Loss: 1.4944 | Val AUC: 0.9764
Epoch 05/10 | Train Loss: 0.6275 | Val Loss: 1.4877 | Val AUC: 0.9764
Epoch 06/10 | Train Loss: 0.6119 | Val Loss: 1.4875 | Val AUC: 0.9764
Epoch 07/10 | Train Loss: 0.6001 | Val Loss: 1.4806 | Val AUC: 0.9766
Epoch 08/10 | Train Loss: 0.5910 | Val Loss: 1.4893 | Val AUC: 0.9752
Epoch 09/10 | Train Loss: 0.5833 | Val Loss: 1.4819 | Val AUC: 0.9762
Epoch 10/10 | Train Loss: 0.5761 | Val Loss: 1.4660 | Val AUC: 0.9766
Learning curves saved to /content/drive/MyDrive/BirdCLEF_Project/repro/outputs/models/tta_ce_10ep/learning_curves_fold4.png